# 05 — Modelagem, otimização e validação

Modelos comparativos: **Random Forest** (*bagging* de árvores profundas: reduz variância) e **XGBoost** (*boosting* de árvores rasas em sequência: reduz viés, com regularização).

Protocolo, todo dentro de `src/train.py`:

1. **Janela de treino** escolhida na validação de 2024.
2. **Busca aleatória** de hiperparâmetros: treino 2019-2023, avaliação em 2024. Critério: F1-macro após ajuste de limiar.
3. **Validação cruzada temporal** (origem móvel, 4 folds anuais) com os parâmetros escolhidos, para medir a estabilidade.

Com `RETREINAR = True` as etapas são executadas de novo (~1 h em 2 núcleos).

In [1]:
# Configuração: roda a partir da raiz do repositório
import os, sys, warnings
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
from src import config

In [2]:
RETREINAR = False
from src import data_processing as dp, train
base = dp.carregar_base()
if RETREINAR:
    train.experimento_janela(base)
    for m in ("xgb", "lr", "rf"):
        train.busca(base, m)
    train.validacao_cruzada_temporal(base)

## 1. Janela de treino

In [3]:
pd.read_csv(config.TABLES_DIR / "janela_treino.csv").round(3)

,inicio_treino,n_treino,f1_macro_argmax,pr_auc_fatais,roc_auc_macro,f1_macro_ajustado,pct_prev_sem_vitimas_argmax,pct_real_sem_vitimas,segundos
0,2017,486737,0.551,0.349,0.777,0.553,12.222,16.101,42.954
1,2019,327879,0.555,0.346,0.782,0.559,9.574,16.101,28.231
2,2021,196885,0.555,0.339,0.781,0.558,9.833,16.101,18.541


Treinar desde 2017 inclui os anos de transição da declaração online, quando a classe Sem Vítimas era o dobro da atual. Começar em 2019 teve o melhor F1-macro na validação, com metade do custo de 2017. Começar em 2021 ficou muito próximo, mas com menos dados.

## 2. Busca de hiperparâmetros

In [4]:
for m in ("xgb", "rf"):
    b = pd.read_csv(config.TABLES_DIR / f"busca_{m}.csv")
    display(Markdown(f"**{m.upper()}** — {len(b)} configurações"))
    display(b.drop(columns=["config"]).round(3).head(8))

**XGB** — 12 configurações

,subsample,reg_lambda,pesos,n_estimators,min_child_weight,max_depth,learning_rate,colsample_bytree,f1_macro_argmax,pr_auc_fatais,roc_auc_macro,f1_macro_ajustado,w_sem,w_fatais,segundos
0,0.7,5.0,balanceado,500,5,8,0.03,0.8,0.486,0.345,0.777,0.561,0.569,0.324,49.2
1,0.7,5.0,nenhum,800,20,8,0.03,0.8,0.461,0.347,0.784,0.561,2.236,2.847,83.7
2,0.9,1.0,nenhum,300,1,8,0.03,1.0,0.447,0.347,0.785,0.561,2.627,3.085,31.0
3,0.7,5.0,nenhum,800,1,6,0.05,0.6,0.463,0.349,0.784,0.560,2.423,2.847,82.5
4,0.7,5.0,nenhum,500,1,6,0.03,0.6,0.451,0.349,0.784,0.560,2.627,3.085,37.3
5,0.7,5.0,raiz,300,20,6,0.05,0.8,0.554,0.348,0.783,0.559,1.273,0.851,24.8
6,0.9,5.0,balanceado,500,1,6,0.03,0.8,0.474,0.347,0.776,0.559,0.569,0.299,36.6
7,0.7,5.0,balanceado,500,5,6,0.03,0.8,0.476,0.347,0.776,0.558,0.617,0.276,38.3


**RF** — 6 configurações

,pesos,n_estimators,min_samples_leaf,max_features,max_depth,f1_macro_argmax,pr_auc_fatais,roc_auc_macro,f1_macro_ajustado,w_sem,w_fatais,segundos
0,balanceado,200,10,0.3,NaN,0.505,0.327,0.768,0.553,0.725,0.447,89.5
1,balanceado,200,10,0.3,30.0,0.506,0.327,0.768,0.552,0.725,0.447,88.2
2,raiz,200,10,0.3,30.0,0.541,0.326,0.771,0.552,1.273,1.084,104.9
3,raiz,200,5,sqrt,20.0,0.523,0.323,0.762,0.545,1.380,1.175,62.9
4,nenhum,200,10,sqrt,NaN,0.371,0.321,0.762,0.544,2.627,3.344,62.2
5,nenhum,200,25,sqrt,30.0,0.347,0.322,0.761,0.543,2.847,3.085,55.1


In [5]:
json.loads((config.TABLES_DIR / "melhores_params.json").read_text())

{'xgb': {'subsample': 0.7,
  'reg_lambda': 5.0,
  'pesos': 'balanceado',
  'n_estimators': 500,
  'min_child_weight': 5,
  'max_depth': 8,
  'learning_rate': 0.03,
  'colsample_bytree': 0.8},
 'lr': {'pesos': 'nenhum', 'C': 10.0},
 'rf': {'pesos': 'balanceado',
  'n_estimators': 200,
  'min_samples_leaf': 10,
  'max_features': 0.3,
  'max_depth': None}}

## 3. Validação cruzada temporal (cenário A)

In [6]:
cv = pd.read_csv(config.TABLES_DIR / "cv_temporal.csv")
display(cv.round(3))
cv.groupby("modelo")[["f1_macro_argmax", "pr_auc_fatais", "roc_auc_macro"]].agg(["mean", "std"]).round(3)

,modelo,treino,validacao,f1_macro_argmax,pr_auc_fatais,roc_auc_macro
0,lr,2019-2020,2021,0.407,0.296,0.732
1,lr,2019-2021,2022,0.442,0.286,0.745
2,lr,2019-2022,2023,0.450,0.312,0.757
3,lr,2019-2023,2024,0.453,0.320,0.766
4,rf,2019-2020,2021,0.491,0.295,0.732
5,rf,2019-2021,2022,0.484,0.294,0.740
6,rf,2019-2022,2023,0.496,0.312,0.756
7,rf,2019-2023,2024,0.505,0.327,0.768
8,xgb,2019-2020,2021,0.478,0.310,0.735
9,xgb,2019-2021,2022,0.473,0.313,0.752


f1_macro_argmax        pr_auc_fatais        roc_auc_macro       
                  mean    std          mean    std          mean    std
modelo                                                                 
lr               0.438  0.021         0.304  0.015         0.750  0.014
rf               0.494  0.009         0.307  0.016         0.749  0.016
xgb              0.478  0.006         0.324  0.016         0.758  0.019

Nos folds a métrica é o F1-macro com `argmax`, sem ajuste de limiar, porque o limiar é escolhido só na validação final. A PR-AUC e a ROC-AUC não dependem de limiar. O desvio-padrão entre anos indica a estabilidade de cada modelo.

## 4. Resumo da validação 2024 por modelo e cenário

In [7]:
val = json.loads((config.TABLES_DIR / "resumo_validacao.json").read_text())
pd.DataFrame({k: v["validacao_2024"] | {"w_sem": v["pesos_limiar"][0], "w_fatais": v["pesos_limiar"][2]} for k, v in val.items()}).T.round(3)

,f1_macro_argmax,pr_auc_fatais,roc_auc_macro,f1_macro_ajustado,w_sem,w_fatais
dummy_A,0.290,0.071,0.500,0.290,1.000,1.000
lr_A,0.453,0.320,0.766,0.543,2.627,2.847
rf_A,0.505,0.327,0.768,0.553,0.725,0.447
xgb_A,0.486,0.345,0.777,0.561,0.569,0.324
dummy_B,0.290,0.071,0.500,0.290,1.000,1.000
lr_B,0.304,0.174,0.661,0.436,3.344,5.000
rf_B,0.414,0.176,0.661,0.438,0.786,0.617
xgb_B,0.397,0.186,0.666,0.444,0.786,0.525
xgb_C,0.507,0.359,0.791,0.576,0.617,0.324
xgb_A_pessoas,0.501,0.351,0.792,0.573,0.525,0.324
